# 02. Cell Type Annotation

**목적**: CellTypist 자동 분류 후 마커 유전자로 검증 및 수동 보정

**입력**: `output/checkpoints/{dataset}/07_clustered_clean.h5ad`  
**출력**: `output/checkpoints/{dataset}/08_annotated.h5ad`


In [ ]:
import sys
sys.path.insert(0, "../../")
sys.path.insert(0, "../")

import scanpy as sc
import pandas as pd
import yaml

from modules.io import load_adata, save_adata
from modules.annotation import run_celltypist, apply_manual_annotation, score_cell_types
from modules.plotting import marker_dotplot, umap_panel

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
DATASET = "human_genes_only"
CHECKPOINT_DIR = f"../../output/checkpoints/{DATASET}"

# markers.yaml 로드
with open("../../configs/markers.yaml") as f:
    markers = yaml.safe_load(f)

adata = load_adata(f"{CHECKPOINT_DIR}/07_clustered_clean.h5ad")
print(adata)

## 1. CellTypist 자동 분류

In [ ]:
# CellTypist 실행
# adata는 log-normalized 상태여야 함 (adata.raw에 원본 보존)
adata = run_celltypist(
    adata,
    model="Immune_All_Low.pkl",   # 인간 면역 atlas 모델
    majority_voting=True,
    over_clustering="leiden",
)
print(adata.obs["celltypist_cell_type"].value_counts())

In [ ]:
# CellTypist 결과 UMAP
umap_panel(
    adata,
    color=["leiden", "celltypist_cell_type", "celltypist_conf_score"],
    ncols=3,
)

## 2. 마커 유전자 Dotplot으로 검증

In [ ]:
# T cell 마커 dotplot (클러스터별)
marker_dotplot(adata, markers["T_cells"], groupby="leiden")

In [ ]:
# B cell + Myeloid 마커
marker_dotplot(adata, {**markers["B_cells"], **markers["Myeloid"]}, groupby="leiden")

In [ ]:
# 마커 기반 점수 계산
adata = score_cell_types(adata, markers)

## 3. 수동 Annotation

위 dotplot과 CellTypist 결과를 참고해서 클러스터별 cell type을 직접 지정하세요.

In [ ]:
# ============================================================
# 수동 설정: 클러스터 → cell type 매핑
# CellTypist 결과와 마커 dotplot을 확인 후 채우세요
# ============================================================
cluster_to_celltype = {
    # "0": "CD4 T",
    # "1": "CD8 T",
    # "2": "NK",
    # "3": "B naive",
    # 나머지 클러스터 추가...
}

if cluster_to_celltype:
    adata = apply_manual_annotation(adata, cluster_to_celltype, cluster_key="leiden", output_key="cell_type")
    print(adata.obs["cell_type"].value_counts())
else:
    # 수동 annotation 전까지 CellTypist 결과 사용
    adata.obs["cell_type"] = adata.obs["celltypist_cell_type"]
    print("Using CellTypist annotations as cell_type")

In [ ]:
# 최종 annotation UMAP
sc.pl.umap(adata, color="cell_type", legend_loc="right margin", title="Cell Types")

In [ ]:
# 저장
out_path = f"{CHECKPOINT_DIR}/08_annotated.h5ad"
save_adata(adata, out_path)
print(f"Saved: {out_path}")